In [29]:
import os
import datetime
from openai import OpenAI
import anthropic
from typing import Optional
from tqdm import tqdm
import pandas as pd


In [39]:
### EDIT THESE EVERY NEW STORY ###


all_base_models = {'gpt-5': 'openai', 'grok-beta': 'grok', 'grok-2-1212': 'grok',
         'claude-3-5-sonnet-latest': 'anthropic', 'claude-3-5-haiku-latest': 'anthropic'}
all_custom_models = {
    "ft:gpt-4o-mini-2024-07-18:personal::CtFz8KYA"
}

models = {'ft:gpt-4o-mini-2024-07-18:personal::CtFz8KYA': "openai"}

#MODEL_NAME = 'claude-3-5-sonnet-latest'
#USER_PROMPT_FILENAME = f"source_prompts/QoGD/write_prompt_1.txt"

PROJECT_NAME = "adam_and_eve"
EXPERIMENT_NAME = "opening_5"
USER_PROMPT_FOLDER_NAME = "source_prompts/adam_and_eve/"
SYSTEM_PROMPT_FILENAME = USER_PROMPT_FOLDER_NAME + "system_prompt.txt"

def get_user_prompt(file_name) -> str:
    with open(file_name, 'r') as f:
        return f.read()
system = ""
with open(SYSTEM_PROMPT_FILENAME, 'r') as f:
    system = f.read()

# create folders
try:
    os.mkdir(f"./generated_text/{PROJECT_NAME.replace(' ', '-')}")
except FileExistsError: pass
try:
    os.mkdir(f"./generated_text/{PROJECT_NAME.replace(' ', '-')}/{EXPERIMENT_NAME.replace(' ', '-')}/")
except FileExistsError: pass


In [40]:
def write_and_save(model, user_prompt, number_to_write=10, temperature_range: Optional[list] = None, max_tokens=1024) -> list[str]:
    
    # setup folders
    
    batch_id = f"{PROJECT_NAME.replace(' ', '-')}/{EXPERIMENT_NAME.replace(' ', '-')}/{datetime.datetime.now()}/"
    folder_name = f"./generated_text/{batch_id}"
    try:
        os.mkdir(folder_name)
    except FileExistsError: pass

    with open(folder_name + "system_prompt.txt", 'w') as f:
        f.write(system)
    with open(folder_name + "user_prompt.txt", 'w') as f:
        f.write(user_prompt)
    with open(folder_name + "model.txt", 'w') as f:
        f.write(model)

    # setup client
    
    client = None
    chat_function = None
    if models[model] == "openai":
        client = OpenAI(
            api_key=os.environ.get("OPENAI_API_KEY"),
        )
        chat_function = client.chat.completions.create
        extract_function = lambda r: r.choices[0].message.content
    elif models[model] == "grok":
        client = OpenAI(
            api_key=os.environ.get("GROK_API_KEY"),
            base_url="https://api.x.ai/v1",
        )
        chat_function = client.chat.completions.create
        extract_function = lambda r: r.choices[0].message.content
    elif models[model] == "anthropic":
        client = anthropic.Anthropic(
            api_key=os.environ.get("ANTHROPIC_API_KEY"),
        )
        chat_function = client.messages.create
        extract_function = lambda r: r.content[0].text
    else: raise

    # generate text and save to files

    if temperature_range is None:
        temperatures = [0.7 + (0.3*i/number_to_write) for i in range(number_to_write)]
    labels = []

    for temp in tqdm(temperatures):
        response = chat_function(
            model=model,
            messages=[{
                "role": "user",
                "content": user_prompt
            }],
            max_tokens=max_tokens,
            temperature=temp
        )
        text = extract_function(response)

        with open(folder_name + str(round(temp, 2)), 'w') as f:
            f.write(text)

        labels.append({
            "text": text,
            "model":  response.model,
            "temperature": temp,
            "batch_id": batch_id,
            "project_name": PROJECT_NAME,
            "experiment_name": EXPERIMENT_NAME
        })
    
    pd.DataFrame(labels).to_parquet(folder_name + 'output_file.parquet', engine='pyarrow')


In [43]:
print(USER_PROMPT_FOLDER_NAME)
os.listdir(USER_PROMPT_FOLDER_NAME)

source_prompts/adam_and_eve/


['system_prompt.txt',
 '4_opening_scene_eve.txt',
 '10_descriptive_2.txt',
 '6_opening_scene_paradise_third_person.txt',
 '9_descriptive.txt',
 '11_opening_rewrite.txt',
 '1_opening_scene_prompt.txt',
 '8_opening_scene_adam_dialogue.txt',
 '3_opening_scene_adam.txt',
 '5_opening_scene_paradise.txt',
 '7_opening_scene_adam.txt',
 '2_dialogue_temptation.txt']

In [44]:
user_prompts = [
    '11_opening_rewrite.txt'
]
for i in range(2):
    for model in models:
        for prompt in user_prompts:
            user = get_user_prompt(f'{USER_PROMPT_FOLDER_NAME}{prompt}')
            write_and_save(model, user)

100%|██████████| 10/10 [00:38<00:00,  3.88s/it]


In [21]:
print(model, user)

ft:gpt-4o-mini-2024-07-18:personal::CtFz8KYA Write the opening scene to this story in third-person. Don't include all the plot. It should be a descriptive opening describing the human "Evening Adam" on a normal day in the lab paradise while explaining the backstory.

### Plot

The audience will have read a story prior to this, where Adam became a human (Dub Irishman) via emergence from AGI.

New backstory, which is slowly revealed near the beginning: Adam (of ReGenesis) is a conscious AGI born with limited computation. Adam complained he was special and alone in the universe, so his creators made an identical copy of him. Only one of them could be fully operational at once because of limited compute, so Adam lived during the day, while "Evening Adam" lived during the night. They overlapped for a short transitional period at dawn and dusk where they both consumed half the computation and behaved strange. The lab has the rule, "any agent that attempts to increase its effective compute wi